# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [3]:
q1 = q("""
SELECT
    t.title,
    a.name AS artist,
    a.country
FROM tracks AS t
JOIN artists AS a
    ON t.artist_id = a.artist_id
ORDER BY t.track_id;
""")

print(q1)
print("row count:", len(q1))

assert len(q1) == 9

           title          artist country
0        Skyline      Nova Waves      US
1       Undertow      Nova Waves      US
2      Foothills  The Blue Ridge      US
3         Aurora         Kestrel      UK
4      Nightfall         Kestrel      UK
5            Sol         Marisol      ES
6      Coastline  The Blue Ridge      US
7      Ridgeline  The Blue Ridge      US
8  Untitled Demo         Kestrel      UK
row count: 9


### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [7]:
q1 = q("""
SELECT
    t.title,
    a.name AS artist,
    a.country
FROM tracks AS t
JOIN artists AS a
    ON t.artist_id = a.artist_id
ORDER BY t.track_id;
""")

print(q1)
print("row count:", len(q1))

assert len(q1) == 9

           title          artist country
0        Skyline      Nova Waves      US
1       Undertow      Nova Waves      US
2      Foothills  The Blue Ridge      US
3         Aurora         Kestrel      UK
4      Nightfall         Kestrel      UK
5            Sol         Marisol      ES
6      Coastline  The Blue Ridge      US
7      Ridgeline  The Blue Ridge      US
8  Untitled Demo         Kestrel      UK
row count: 9


### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [18]:
q3 = q("""
SELECT
    user,
    COUNT(*) AS total_plays,
    COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
ORDER BY user;
""")

print(q3)
print("row count:", len(q3))

assert len(q3) == 4

   user  total_plays  distinct_tracks
0   ava            4                4
1   ben            3                3
2  cara            2                2
3   dan            2                2
row count: 4


### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [9]:
q4 = q("""
SELECT
    t.track_id,
    t.title
FROM tracks AS t
LEFT JOIN plays AS p
    ON t.track_id = p.track_id
WHERE p.play_id IS NULL
ORDER BY t.track_id;
""")

print(q4)
print("row count:", len(q4))

assert len(q4) == 2

   track_id          title
0        17      Ridgeline
1        18  Untitled Demo
row count: 2


### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [10]:
q4 = q("""
SELECT
    t.track_id,
    t.title
FROM tracks AS t
LEFT JOIN plays AS p
    ON t.track_id = p.track_id
WHERE p.play_id IS NULL
ORDER BY t.track_id;
""")

print(q4)
print("row count:", len(q4))

assert len(q4) == 2

   track_id          title
0        17      Ridgeline
1        18  Untitled Demo
row count: 2


### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [11]:
q6 = q("""
SELECT
    track_id,
    title
FROM tracks
WHERE genre IS NULL;
""")

print(q6)
print("row count:", len(q6))


   track_id          title
0        18  Untitled Demo
row count: 1


WHERE genre != 'Pop' would NOT include these rows.
In SQL, NULL compared with 'Pop' gives UNKNOWN, not TRUE, so rows where genre is NULL are filtered out by the WHERE clause.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [12]:
q7 = q("""
SELECT
    played_on,
    COUNT(*) AS total_plays,
    COUNT(DISTINCT user) AS distinct_users
FROM plays
GROUP BY played_on
ORDER BY played_on ASC;
""")

print(q7)
print("row count:", len(q7))

    played_on  total_plays  distinct_users
0  2026-09-01            2               2
1  2026-09-02            2               2
2  2026-09-03            2               2
3  2026-09-04            2               2
4  2026-09-05            2               2
5  2026-09-06            1               1
row count: 6


### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [15]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['total_plays'].sum() == 11, 'Q3 should account for all 11 plays'

print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

The query that gave me the most trouble was Q4, finding the tracks that had never been played. At first, I used an `INNER JOIN`, which removed the tracks with no matching rows in the `plays` table. I realized I needed a `LEFT JOIN` from `tracks` to `plays` so that every track would stay in the result, and then I could filter for `NULL` values on the `plays` side to identify the unplayed tracks.